In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install python-calamine

import pandas as pd, numpy as np
BASE  = '/content/drive/MyDrive/Palestine_Project'
RAW   = f'{BASE}/00_raw/Palestine_Project_Data.xlsx'
CLEAN = f'{BASE}/01_clean'
ML    = f'{BASE}/02_ml'

def read_sheet(name, **kw):
    return pd.read_excel(RAW, sheet_name=name, engine='calamine', **kw)

def save_clean(df, filename):
    df.to_csv(f'{CLEAN}/{filename}', index=False, encoding='utf-8')
    print(f'saved {filename}: {len(df):,} rows x {df.shape[1]} cols')

In [ ]:
import ast
b = read_sheet('2_Buildings_Damaged_Daily'); b['report_date'] = pd.to_datetime(b['report_date'])

spec = {('civic_buildings',       'ext_destroyed'):         ('civic',       'destroyed'),
        ('educational_buildings', 'ext_destroyed'):         ('education',   'destroyed'),
        ('educational_buildings', 'ext_damaged'):           ('education',   'damaged'),
        ('places_of_worship',     'ext_mosques_destroyed'): ('mosque',      'destroyed'),
        ('places_of_worship',     'ext_mosques_damaged'):   ('mosque',      'damaged'),
        ('places_of_worship',     'ext_churches_destroyed'):('church',      'destroyed'),
        ('residential',           'ext_destroyed'):         ('residential', 'destroyed')}
parts = []
for (col, key), (cat, status) in spec.items():
    s = b[col].map(lambda x: ast.literal_eval(x).get(key))
    t = pd.DataFrame({'date_key': b['report_date'].dt.strftime('%Y%m%d').astype(int),
                      'category': cat, 'status': status, 'count_cum': s})
    t = t[t['count_cum'].notna()].copy()
    t['count_cum']  = t['count_cum'].astype('int64')
    t['count_daily'] = t['count_cum'].diff().fillna(t['count_cum']).astype('int64')
    t['is_decrease_flag'] = (t['count_daily'] < 0).astype(int)
    parts.append(t)
B = pd.concat(parts, ignore_index=True)

print('rows', len(B), '(expected 5,081) | decreases', int(B['is_decrease_flag'].sum()), '(one expected: mosques damaged)')
print(B[B['date_key'] == 20251005][['category','status','count_cum']].to_string(index=False))
save_clean(B, 'fact_buildings_damaged_daily.csv')